# Lognormal 右偏重尾误差下的 DML 稳健性模拟（含正态基准，$p=5$）

本 notebook 与 `DML_p5_student_t_heavytail.ipynb` 保持相同的样本量、DGP、交叉拟合、学习器、重复次数、随机种子和评价指标，只将结果方程误差改为 lognormal。模型为
$$Y=D\theta_0+g_0(X)+U,\qquad D=m_0(X)+V.$$

令 $W=\exp(\sigma Z)$、$Z\sim N(0,1)$，并使用
$$U=\frac{W-E(W)}{\sqrt{\operatorname{Var}(W)}}
=\frac{\exp(\sigma Z)-\exp(\sigma^2/2)}{\sqrt{(\exp(\sigma^2)-1)\exp(\sigma^2)}}.$$
因此每个 lognormal 场景理论上均满足 $E(U)=0$、$\operatorname{Var}(U)=1$。注意：均值中心化后的误差仍然右偏，且中位数不为 0；这正是与对称 Student-t 情形的重要区别。

In [ ]:
# ============================================================
# 调参区：除分布专属参数外，与 Student-t notebook 保持一致
# ============================================================
P = 5
N = 1000
THETA_0 = 1.0

# 7 个 lognormal 场景对应原 Student-t notebook 的 7 个重尾场景；
# sigma 越大，右偏和尾部越强。按“重尾 -> 轻尾”排列，最后加入 N(0,1) baseline。
LOGNORMAL_SIGMA_GRID = [2.0, 1.5, 1.25, 1.0, 0.75, 0.5, 0.25]
NORMAL_BASELINE = "N(0,1)"
ERROR_GRID = [*LOGNORMAL_SIGMA_GRID, NORMAL_BASELINE]
DGP_NAMES = ["DGP1_linear", "DGP2_nonlinear"]

N_FOLDS = 5
N_REPLICATIONS = 1000
BASE_SEED = 20260826
N_JOBS = -1

# 先设为 True 验证；正式计算时改为 False。快速模式不会写出 CSV。
QUICK_TEST = False
QUICK_N = 300
QUICK_REPLICATIONS = 2

BOOSTING_PARAMS = {
    "learning_rate": 0.05,
    "max_iter": 100,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 20,
    "l2_regularization": 1.0,
}

# 正式运行后，汇总结果默认保存为桌面上的一个 CSV 文件。
SAVE_RESULTS = True
OUTPUT_CSV = None  # 导入依赖后自动设为：桌面/dml_p5_lognormal_summary.csv

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")

if OUTPUT_CSV is None:
    OUTPUT_CSV = Path.home() / "Desktop" / "dml_p5_lognormal_summary.csv"
else:
    OUTPUT_CSV = Path(OUTPUT_CSV)

assert P == 5, "本 notebook 专门用于 p=5；如改维度，请同步检查 DGP 公式。"
assert all(sigma > 0 for sigma in LOGNORMAL_SIGMA_GRID), "lognormal sigma 必须大于 0。"
assert len(ERROR_GRID) == len(set(ERROR_GRID)), "误差场景不能重复。"
print(f"依赖与参数检查完成。正式结果将保存到：{OUTPUT_CSV}")

## 1. 两种 DGP 与中心化、单位方差化的 lognormal 误差

协变量结构、nuisance functions、$V\sim N(0,1)$ 和真实参数均与 Student-t notebook 相同。各 lognormal 场景和正态基准均具有理论均值 0、方差 1，使差异主要来自偏度与尾部形态。

In [ ]:
def ar1_covariance(p: int, rho: float = 0.5) -> np.ndarray:
    index = np.arange(p)
    return rho ** np.abs(index[:, None] - index[None, :])


def nuisance_functions(X: np.ndarray, dgp_name: str):
    x1, x2, x3, x4, x5 = X.T
    if dgp_name == "DGP1_linear":
        m0 = 0.80*x1 - 0.50*x2 + 0.30*x3 + 0.20*x4 - 0.10*x5
        g0 = 1.00*x1 + 0.50*x2 - 0.50*x3 + 0.25*x4 + 0.10*x5
    elif dgp_name == "DGP2_nonlinear":
        m0 = (0.60*np.sin(x1) + 0.35*(x2**2 - 1.0)
              + 0.25*x3*x4 - 0.20*np.tanh(x5))
        g0 = (1.20*np.sin(x1) + 0.70*(x2**2 - 1.0)
              + 0.50*x3*x4 + 0.30*np.tanh(x5))
    else:
        raise ValueError(f"未知 DGP: {dgp_name}")
    return m0, g0


def standardized_centered_lognormal(rng: np.random.Generator, sigma: float, size: int):
    """生成理论均值 0、方差 1 的右偏 lognormal 误差。"""
    if sigma <= 0:
        raise ValueError("lognormal sigma 必须大于 0。")
    z = rng.normal(0.0, 1.0, size=size)
    w = np.exp(sigma * z)
    mean_w = np.exp(0.5 * sigma**2)
    sd_w = np.sqrt(np.expm1(sigma**2) * np.exp(sigma**2))
    return (w - mean_w) / sd_w


def generate_data(n: int, error_case, dgp_name: str, seed: int, theta0: float = THETA_0):
    rng = np.random.default_rng(seed)
    sigma_x = ar1_covariance(P, rho=0.5)
    X = rng.multivariate_normal(np.zeros(P), sigma_x, size=n)
    m0, g0 = nuisance_functions(X, dgp_name)
    V = rng.normal(0.0, 1.0, size=n)
    if error_case == NORMAL_BASELINE:
        U = rng.normal(0.0, 1.0, size=n)
    else:
        U = standardized_centered_lognormal(rng, float(error_case), n)
    D = m0 + V
    Y = theta0 * D + g0 + U
    return X, D, Y, U


X_demo, D_demo, Y_demo, U_demo = generate_data(
    N, LOGNORMAL_SIGMA_GRID[0], DGP_NAMES[0], BASE_SEED
)
print(
    f"X shape: {X_demo.shape}; sigma={LOGNORMAL_SIGMA_GRID[0]:g}; "
    f"U mean={U_demo.mean():.3f}; U variance={U_demo.var(ddof=1):.3f}; "
    f"max(U)={U_demo.max():.2f}"
)

## 2. Cross-fitted Mean DML 与 Median DML

估计器实现与 Student-t notebook 保持一致：Mean DML 使用条件均值学习器和残差 OLS；Median DML 使用条件中位数学习器和 LAD 斜率。所有 nuisance 预测均为 out-of-fold 预测。

In [ ]:
def make_nuisance_model(kind: str, seed: int):
    common = dict(BOOSTING_PARAMS)
    common["random_state"] = seed
    if kind == "mean":
        return HistGradientBoostingRegressor(loss="squared_error", **common)
    if kind == "median":
        return HistGradientBoostingRegressor(loss="quantile", quantile=0.5, **common)
    raise ValueError("kind 必须是 'mean' 或 'median'。")


def cross_fitted_residuals(X, D, Y, kind: str, n_folds: int, seed: int):
    d_hat = np.empty_like(D, dtype=float)
    y_hat = np.empty_like(Y, dtype=float)
    splitter = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for fold, (train_idx, test_idx) in enumerate(splitter.split(X)):
        model_d = make_nuisance_model(kind, seed + 100*fold + 1)
        model_y = make_nuisance_model(kind, seed + 100*fold + 2)
        model_d.fit(X[train_idx], D[train_idx])
        model_y.fit(X[train_idx], Y[train_idx])
        d_hat[test_idx] = model_d.predict(X[test_idx])
        y_hat[test_idx] = model_y.predict(X[test_idx])
    return D - d_hat, Y - y_hat


def weighted_median(values: np.ndarray, weights: np.ndarray) -> float:
    order = np.argsort(values)
    values_sorted = values[order]
    weights_sorted = weights[order]
    cutoff = 0.5 * weights_sorted.sum()
    index = np.searchsorted(np.cumsum(weights_sorted), cutoff, side="left")
    return float(values_sorted[min(index, len(values_sorted)-1)])


def fit_mean_dml(d_res: np.ndarray, y_res: np.ndarray):
    jacobian = np.mean(d_res**2)
    if jacobian <= 1e-12:
        return np.nan, np.nan
    theta_hat = np.mean(d_res*y_res) / jacobian
    score = d_res * (y_res - theta_hat*d_res)
    influence = score / jacobian
    se_hat = np.sqrt(np.var(influence, ddof=1) / len(d_res))
    return float(theta_hat), float(se_hat)


def fit_median_dml(d_res: np.ndarray, y_res: np.ndarray):
    keep = np.abs(d_res) > 1e-10
    if not np.any(keep):
        return np.nan, np.nan
    ratios = y_res[keep] / d_res[keep]
    weights = np.abs(d_res[keep])
    theta_hat = weighted_median(ratios, weights)
    residual = y_res - theta_hat*d_res
    n = len(residual)
    mad = np.median(np.abs(residual - np.median(residual))) / 0.67448975
    robust_scale = max(float(mad), 1e-6)
    bandwidth = max(1.06*robust_scale*n**(-1.0/5.0), 1e-6)
    kernel_at_zero = norm.pdf(residual / bandwidth) / bandwidth
    A_hat = np.mean((d_res**2) * kernel_at_zero)
    B_hat = 0.25 * np.mean(d_res**2)
    se_hat = np.sqrt(B_hat / (A_hat**2*n)) if A_hat > 1e-12 else np.nan
    return float(theta_hat), float(se_hat)


def estimate_both_dml(X, D, Y, seed: int):
    estimates = []
    d_mean, y_mean = cross_fitted_residuals(X, D, Y, "mean", N_FOLDS, seed)
    estimates.append(("Mean_DML", *fit_mean_dml(d_mean, y_mean)))
    d_median, y_median = cross_fitted_residuals(X, D, Y, "median", N_FOLDS, seed)
    estimates.append(("Median_DML", *fit_median_dml(d_median, y_median)))
    return estimates

## 3. Monte Carlo 主循环

每个 `(DGP, error_distribution, replication)` 使用确定且互不重叠的随机种子；两种估计量共享同一份样本，以便配对比较。

In [ ]:
def one_replication(dgp_name: str, error_case, replication: int, n_use: int):
    dgp_id = DGP_NAMES.index(dgp_name)
    error_id = ERROR_GRID.index(error_case)
    seed = BASE_SEED + 100_000*dgp_id + 1_000*error_id + replication
    X, D, Y, U = generate_data(n_use, error_case, dgp_name, seed)
    estimates = estimate_both_dml(X, D, Y, seed + 500_000)
    rows = []
    for estimator, theta_hat, se_hat in estimates:
        ci_low = theta_hat - 1.96*se_hat
        ci_high = theta_hat + 1.96*se_hat
        rows.append({
            "dgp": dgp_name,
            "p": P,
            "n": n_use,
            "error_distribution": (NORMAL_BASELINE if error_case == NORMAL_BASELINE
                                   else f"Lognormal(sigma={error_case:g})"),
            "tail_order": error_id,
            "lognormal_sigma": (np.nan if error_case == NORMAL_BASELINE
                                  else float(error_case)),
            "replication": replication,
            "estimator": estimator,
            "theta_hat": theta_hat,
            "se_hat": se_hat,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "covered": bool(ci_low <= THETA_0 <= ci_high),
            "sample_u_mean": np.mean(U),
            "sample_u_variance": np.var(U, ddof=1),
            "sample_max_u": np.max(U),
            "sample_skewness_u": pd.Series(U).skew(),
        })
    return rows


def run_simulation():
    n_use = QUICK_N if QUICK_TEST else N
    r_use = QUICK_REPLICATIONS if QUICK_TEST else N_REPLICATIONS
    tasks = [
        (dgp_name, error_case, replication, n_use)
        for dgp_name in DGP_NAMES
        for error_case in ERROR_GRID
        for replication in range(r_use)
    ]
    mode = "QUICK_TEST" if QUICK_TEST else "FORMAL"
    print(
        f"开始运行 [{mode}]：p={P}, n={n_use}, DGP={len(DGP_NAMES)}, "
        f"error scenarios={ERROR_GRID}, repetitions={r_use}"
    )
    start = time.perf_counter()
    nested_rows = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(one_replication)(*task) for task in tasks
    )
    rows = [row for pair in nested_rows for row in pair]
    print(f"完成，用时 {time.perf_counter()-start:.1f} 秒，共 {len(rows)} 条估计结果。")
    return pd.DataFrame(rows)


results = run_simulation()
results.head()

## 4. 汇总指标并保存一个 CSV

CSV 每行对应一个 `(DGP, error_distribution, estimator)` 场景，包含 Bias、SD、平均 SE、RMSE、95% coverage 和误差样本诊断。

In [ ]:
def summarize_simulation(results_df: pd.DataFrame) -> pd.DataFrame:
    work = results_df.copy()
    work["error"] = work["theta_hat"] - THETA_0
    work["squared_error"] = work["error"]**2
    summary = (
        work.groupby(
            ["dgp", "error_distribution", "tail_order", "estimator"],
            as_index=False,
        )
        .agg(
            lognormal_sigma=("lognormal_sigma", "first"),
            mean_estimate=("theta_hat", "mean"),
            bias=("error", "mean"),
            sd=("theta_hat", "std"),
            mean_se=("se_hat", "mean"),
            mse=("squared_error", "mean"),
            coverage=("covered", "mean"),
            mean_sample_u_mean=("sample_u_mean", "mean"),
            mean_sample_u_variance=("sample_u_variance", "mean"),
            mean_sample_max_u=("sample_max_u", "mean"),
            mean_sample_skewness_u=("sample_skewness_u", "mean"),
            valid_runs=("theta_hat", "count"),
        )
    )
    summary["rmse"] = np.sqrt(summary.pop("mse"))
    return summary.sort_values(["dgp", "tail_order", "estimator"]).reset_index(drop=True)


summary = summarize_simulation(results)
display(summary.round({
    "mean_estimate": 4, "bias": 4, "sd": 4, "mean_se": 4,
    "rmse": 4, "coverage": 3, "mean_sample_u_mean": 3,
    "mean_sample_u_variance": 3, "mean_sample_max_u": 2,
    "mean_sample_skewness_u": 2,
}))

if SAVE_RESULTS and not QUICK_TEST:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"正式汇总结果已保存：{OUTPUT_CSV.resolve()}")
elif QUICK_TEST:
    print("QUICK_TEST=True：验证模式不写出 CSV，避免把测试结果误当作正式结果。")
else:
    print("SAVE_RESULTS=False：未写出 CSV。")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_specs = [("bias", "Bias"), ("rmse", "RMSE"), ("coverage", "95% CI coverage")]
for ax, (metric, label) in zip(axes, plot_specs):
    sns.lineplot(
        data=summary, x="tail_order", y=metric, hue="estimator",
        style="dgp", markers=True, dashes=True, ax=ax,
    )
    ax.set_xticks(range(len(ERROR_GRID)))
    ax.set_xticklabels(
        [f"LN({x:g})" if x != NORMAL_BASELINE else NORMAL_BASELINE for x in ERROR_GRID],
        rotation=30,
    )
    ax.set_xlabel("Error distribution (left = heavier right tail)")
    ax.set_ylabel(label)
    if metric == "bias":
        ax.axhline(0.0, color="black", linewidth=1, linestyle=":")
    if metric == "coverage":
        ax.axhline(0.95, color="black", linewidth=1, linestyle=":")
        ax.set_ylim(0.0, 1.02)
plt.tight_layout()
plt.show()

## 使用说明

1. 调试时把 `QUICK_TEST=True`，从头运行；快速模式不会写 CSV。
2. 正式实验保持 `QUICK_TEST=False`，当前使用 `N=1000`、`N_REPLICATIONS=1000`，完成后自动把汇总结果保存到桌面的 `dml_p5_lognormal_summary.csv`。
3. `sigma` 越大，lognormal 的右偏和尾部越强；`sigma=2` 的有限样本波动可能非常大。
4. 所有 lognormal 误差都按总体公式做均值中心化和单位方差化，不使用样本均值/样本方差标准化。
5. 因为 lognormal 不对称，均值中心化后中位数不为 0。解释 Mean DML 与 Median DML 的差异时，应把这种目标函数差异与重尾效应一并说明。